In [262]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd


verbose = False

In [263]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


In [264]:

def learnQ(targets, covariates,embedding_dim,verbose):
    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)


    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.1
        lambda_l2_w = 1
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        scheduler.step()

        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final
    

## Toy Example

### Toy data set (no underlying patterns)

Notice that the number of time points is greater than the number of units. What if I change this?

In [265]:

# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0], [4.0, 6.0]], dtype=torch.float64)

d_3 = torch.tensor([4.0, 4.0], dtype=torch.float64)
Y_3 = torch.tensor([[1.0, 3.0], [2.0, 4.0]], dtype=torch.float64)

target_vectors = [d_1, d_2, d_3]
covariate_matrices = [Y_1,Y_2,Y_3]


In [266]:
print(target_vectors[0].shape)   # should be (2,)
print(covariate_matrices[0].shape)  # should be (2, 2)

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 10, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(d_3)
print(Y_3@Q_final@w_final)



torch.Size([2])
torch.Size([2, 2])
Step    0 | Loss: 50.60827539
Step  200 | Loss: 20.77173513
Step  400 | Loss: 18.02404691
Step  600 | Loss: 17.79353273
Step  800 | Loss: 17.76607085
Step 1000 | Loss: 17.75367149
Step 1200 | Loss: 17.74840771
Step 1400 | Loss: 17.74308279
Step 1600 | Loss: 17.73898747
Step 1800 | Loss: 17.73610250
tensor([6., 2.], dtype=torch.float64)
tensor([4.27, 3.67], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([2.13, 3.07], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([4.86, 4.27], dtype=torch.float64)


Making the number of donors greater than the number of time points.
This has the effect of greatly decreasing the loss.
And it is also pretty clear that the loss doesn't depend too strongly on the embedding dimension.

In [267]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)



Step    0 | Loss: 74.09506824
Step  200 | Loss: 1.68906200
Step  400 | Loss: 1.14762923
Step  600 | Loss: 0.99151832
Step  800 | Loss: 0.95811428
Step 1000 | Loss: 0.94108430
Step 1200 | Loss: 0.93349423
Step 1400 | Loss: 0.92581964
Step 1600 | Loss: 0.91995205
Step 1800 | Loss: 0.91576849
tensor([6., 2.], dtype=torch.float64)
tensor([5.78, 1.84], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([5.17, 3.20], dtype=torch.float64)


# Synthetic Data Experiment

- Still need to implement standardization


In [ ]:
# should probably normalize the data in advance, or maybe I don't even need to.

# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_timepoints, n_outcomes, n_units)

test = Y[len(Y)-1]
train = Y[0:len(Y)-1]

verbose=False
if (verbose):
    print("Test data: \n\n", test)
    print("\nTrain data: \n\n", train)

# split the data into the target vectors and the covariate matrices
# goal here is to use final outcomes to predict final target
train_target_vectors = [torch.from_numpy(train[i][0:train[i].shape[0],0:1]).squeeze() for i in range(len(train))]
train_covariate_matrices = [torch.from_numpy(train[i][0:train[i].shape[0],1:train[i].shape[1]]) for i in range(len(train))]

verbose=False
if verbose == True:
    for i in range(len(train)):
        print("\nTarget vector", i, ":\n", target_vectors[i])
        print("\nCovariate matrix", i, ":\n", covariate_matrices[i])


# The treated unit at the final time point
test_target_vector = torch.from_numpy(test[0:test.shape[0],0:1]).squeeze()
test_covariate_matrix = torch.from_numpy(test[0:test.shape[0],1:test.shape[1]])

# run the learn Q function:
# NB: embedding into a much lower dimension also works! (try 2)
# embedding dimension doesn't make that much of a difference interestingly
Q_final, w_final = learnQ(train_target_vectors, train_covariate_matrices, 9, False)

torch.set_printoptions(sci_mode=False, precision=2)
print("Original covariates: \n", test_covariate_matrix)
print("\nSynthetic covariates: \n",test_covariate_matrix@Q_final)
print("\nFinal weights: \n", w_final.round(2))
print("\nComputed: \n", test_covariate_matrix@Q_final@w_final)
print("\nGoal: \n", test_target_vector)




Step    0 | Loss: 22.93699479
Step  200 | Loss: 2.51981773
Step  400 | Loss: 1.45244359
Step  600 | Loss: 1.30838141
Step  800 | Loss: 1.27897528
Step 1000 | Loss: 1.26827558
Step 1200 | Loss: 1.26577658
Step 1400 | Loss: 1.26426917
Step 1600 | Loss: 1.26334995
Step 1800 | Loss: 1.26299411
Original covariates: 
 tensor([[ 8.97,  8.54,  2.92,  8.89, 10.87, 11.89, 11.36],
        [ 5.36,  4.97,  3.84,  3.67,  1.48, -2.85, -0.24],
        [ 3.47,  2.31,  5.36,  2.25,  3.32, -2.23, -1.73]],
       dtype=torch.float64)

Synthetic covariates: 
 tensor([[ 7.24, -0.00, -0.00,  0.01,  0.00, -0.00, -0.00,  0.00,  0.00, 11.10,
          0.49, -0.00,  0.00,  0.00,  0.00,  0.00, -0.00, -0.00, -0.00, -0.00],
        [ 1.60,  0.00, -0.00, -0.01, -0.00, -0.00, -0.00, -0.00,  0.00,  2.26,
         -0.07, -0.00, -0.00, -0.00,  0.00, -0.00, -0.00,  0.00, -0.00, -0.00],
        [ 4.52,  0.00, -0.00,  0.00, -0.00, -0.00, -0.00, -0.00, -0.00,  6.47,
          0.55, -0.00, -0.00, -0.00, -0.00, -0.00, -0.00, 